# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umemon254/Fly-rank/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [15]:
HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute("DROP SECRET IF EXISTS hf;")

con.execute(f"""
CREATE SECRET hf (
    TYPE HTTP,
    BEARER_TOKEN '{HF_TOKEN}'
);
""")

In [ ]:
# ============================================================
# W07 ACTION PLAYBOOK - COMPLETE SETUP
# ============================================================

from google.colab import userdata

import os
import duckdb
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

print("Imports complete")


# ============================================================
# 1. Load Hugging Face token
# ============================================================

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found in Colab Secrets. "
        "Add your Hugging Face token as HF_TOKEN and run again."
    )

os.environ["HF_TOKEN"] = HF_TOKEN

print("HF_TOKEN loaded")


# ============================================================
# 2. Connect to DuckDB
# ============================================================

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute("DROP SECRET IF EXISTS hf;")

con.execute(f"""
CREATE SECRET hf (
    TYPE HTTP,
    BEARER_TOKEN '{HF_TOKEN}'
);
""")

print("DuckDB connection ready")


# ============================================================
# 3. Dataset paths
# ============================================================

march_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

april_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-04/*.parquet"
)


# ============================================================
# 4. Build March features
# ============================================================

march = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_clicks) AS clicks_march,

    SUM(gsc_impressions) AS impressions_march,

    SUM(gsc_clicks)
        / NULLIF(SUM(gsc_impressions), 0)
        AS ctr_march,

    SUM(gsc_avg_position * gsc_impressions)
        / NULLIF(SUM(gsc_impressions), 0)
        AS position_march,

    SUM(sessions_organic) AS organic_sessions_march,

    SUM(sessions_ai) AS ai_sessions_march,

    SUM(sessions_paid) AS paid_sessions_march,

    SUM(sessions_referral) AS referral_sessions_march

FROM read_parquet('{march_path}')

WHERE client_has_gsc IS TRUE
  AND gsc_impressions > 0

GROUP BY
    client_hash_id,
    content_hash_id
""").df()


print("March data loaded:", len(march), "rows")


# ============================================================
# 5. Build April outcome data
# ============================================================

april = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_clicks) AS clicks_april,

    SUM(gsc_impressions) AS impressions_april

FROM read_parquet('{april_path}')

WHERE client_has_gsc IS TRUE
  AND gsc_impressions > 0

GROUP BY
    client_hash_id,
    content_hash_id
""").df()


print("April data loaded:", len(april), "rows")


# ============================================================
# 6. Combine March features + April outcome
# ============================================================

df = march.merge(
    april,
    on=[
        "client_hash_id",
        "content_hash_id"
    ],
    how="inner"
)


# ============================================================
# 7. Create the target
# ============================================================

df = df[df["clicks_march"] > 0].copy()

df["is_declining"] = (
    df["clicks_april"]
    <= 0.80 * df["clicks_march"]
).astype(int)


print()
print("Final feature rows:", len(df))
print(
    "Declining rate:",
    round(df["is_declining"].mean(), 3)
)


# ============================================================
# 8. Define model features
# ============================================================

FEATURES = [
    "clicks_march",
    "impressions_march",
    "ctr_march",
    "position_march",
    "organic_sessions_march",
    "ai_sessions_march",
    "paid_sessions_march",
    "referral_sessions_march",
]


# ============================================================
# 9. Clean feature data
# ============================================================

X = (
    df[FEATURES]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y = df["is_declining"].astype(int)


# ============================================================
# 10. Recreate the Week-5 client-level split
# ============================================================

RANDOM_STATE = 42

clients = (
    df["client_hash_id"]
    .drop_duplicates()
    .to_numpy()
)

rng = np.random.default_rng(RANDOM_STATE)

clients = rng.permutation(clients)

test_client_count = max(
    1,
    int(round(len(clients) * 0.20))
)

test_clients = set(
    clients[:test_client_count]
)

test_mask = df["client_hash_id"].isin(
    test_clients
)

train_df = df.loc[~test_mask].copy()

test_df = df.loc[test_mask].copy()


print()
print("Client-level split created")
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print(
    "Train clients:",
    train_df["client_hash_id"].nunique()
)
print(
    "Test clients:",
    test_df["client_hash_id"].nunique()
)


# ============================================================
# 11. Prepare training data
# ============================================================

X_train = (
    train_df[FEATURES]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y_train = (
    train_df["is_declining"]
    .astype(int)
)


# ============================================================
# 12. Recreate Week-5 Logistic Regression model
# ============================================================

logistic = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),

    (
        "model",
        LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=RANDOM_STATE
        )
    )
])


# ============================================================
# 13. Train model
# ============================================================

logistic.fit(
    X_train,
    y_train
)


print()
print("Logistic Regression trained successfully")


# ============================================================
# 14. Final setup check
# ============================================================

required_variables = [
    "df",
    "train_df",
    "test_df",
    "FEATURES",
    "logistic"
]

missing = [
    name
    for name in required_variables
    if name not in globals()
]

if missing:
    raise RuntimeError(
        f"Setup failed. Missing variables: {missing}"
    )

print()
print("======================================")
print("W07 SETUP COMPLETE")
print("======================================")
print("df:", df.shape)
print("train_df:", train_df.shape)
print("test_df:", test_df.shape)
print("Features:", len(FEATURES))
print("Model: Logistic Regression")
print("======================================")

Imports complete
HF_TOKEN loaded
DuckDB connection ready


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March data loaded: 176738 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

April data loaded: 194760 rows

Final feature rows: 68040
Declining rate: 0.61

Client-level split created
Train rows: 60541
Test rows: 7499
Train clients: 34
Test clients: 8

Logistic Regression trained successfully

W07 SETUP COMPLETE
df: (68040, 13)
train_df: (60541, 13)
test_df: (7499, 13)
Features: 8
Model: Logistic Regression


In [ ]:
print("======================================")

In [ ]:
import os
import duckdb

print("HF_TOKEN exists:", bool(HF_TOKEN))

# Tell DuckDB/Hugging Face which token to use
os.environ["HF_TOKEN"] = HF_TOKEN

print("HF_TOKEN environment variable set")

HF_TOKEN exists: True
HF_TOKEN environment variable set


In [ ]:
con = duckdb.connect()

con.execute("""
INSTALL httpfs;
LOAD httpfs;
""")

print("DuckDB httpfs loaded")


DuckDB httpfs loaded


In [ ]:
print("HF_TOKEN:", bool(HF_TOKEN))
print("DuckDB:", con.execute("SELECT version()").fetchone()[0])

HF_TOKEN: True
DuckDB: v1.3.2


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 2. Intended use and limits

This playbook is intended to help a content or SEO reviewer decide which content items deserve attention first.

The output is decision-support. It does not prove that an item will decline and it does not explain the cause of a decline by itself.

The recommended workflow is:

1. Review the highest-ranked items.
2. Check the reason codes and March performance.
3. Inspect the actual content and relevant context.
4. Decide whether a refresh or further investigation is appropriate.
5. Record the human decision separately.

The analysis is based on the available dataset and the defined March-to-April decline label. It should not be treated as a production forecasting system.

The model should not automatically change, publish, delete, redirect, or otherwise modify content.

### Archetype → action mapping

The model score identifies items for review, while the reason codes help suggest what the reviewer should investigate.

| Observed pattern | Suggested review action |
|---|---|
| LOW_CTR | Review title, search intent alignment, and snippet clarity |
| WEAK_POSITION | Review search intent, topical coverage, and on-page relevance |
| LOW_ORGANIC | Check whether the page is receiving enough search visibility |
| LOW_AI | Review whether the content is clearly structured and useful for answer-oriented discovery |
| Multiple reason codes | Prioritize for a broader content review |

These are review suggestions, not automatic fixes.

## Decay / refresh insight

The target in this analysis is a measured decline in April clicks compared with March clicks.

Therefore, the analysis can identify content items that match the defined decline pattern. It cannot by itself establish why the decline happened.

A possible refresh signal is an item with a high model score combined with weak observed performance indicators such as lower CTR or weaker search position.

This should be treated as a directional signal for human review rather than proof that a content refresh will improve performance.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 3. Human review + the no-go list

Every recommended item should receive human review before any content decision is made.

The reviewer should check:

- whether the page still matches its intended search intent;
- whether the information is current;
- whether the title and content accurately represent the page;
- whether the observed performance pattern has a reasonable explanation;
- whether there are important contextual factors not represented in the dataset.

### What should NOT be automated

The model should not automatically:

- rewrite or publish content;
- delete pages;
- create redirects;
- change important SEO settings;
- change client-facing recommendations without review;
- claim that a page will recover after a refresh;
- treat a high model score as proof of causation.

The model output is a prioritization aid. Final decisions remain with a human reviewer.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 4. Monitoring / retrain triggers

The model should be reviewed when the underlying data or business question changes.

Useful monitoring signals include:

- the observed decline rate changes substantially;
- the available features change;
- the meaning of the target label changes;
- model performance falls on later data;
- the distribution of important features changes;
- the ranked queue becomes less useful to human reviewers.

A retraining review should be considered when new labeled periods become available and there is enough comparable data to evaluate the model again.

Retraining should not happen simply because a model is available. The new version should be compared with the existing approach using the same evaluation logic.

## Cost / value thinking

The queue is designed to help reviewers spend limited attention on the items that appear most relevant.

The main value is prioritization: instead of reviewing every content item equally, a reviewer can start with the highest-ranked items.

The cost is reviewer time. A high model score does not remove the need for checking the actual page.

For this reason, the practical value of the playbook depends on whether the ranked queue helps a reviewer identify useful items more efficiently than reviewing items without model-based prioritization.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
import os

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# Export the ranked queue.
output_path = "work/outputs/w07_ranked_action_queue.csv"

ranked_queue.to_csv(
    output_path,
    index=False
)

print(f"Exported ranked queue to: {output_path}")
print(f"Rows exported: {len(ranked_queue)}")

NameError: name 'ranked_queue' is not defined

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.